In [13]:
!pip install ucimlrepo

In [14]:
!pip install tf2onnx

In [15]:
import tf2onnx
import tensorflow as tf

In [23]:
# Citation for the data: Janosi, A., Steinbrunn, W., Pfisterer, M., & Detrano, R. (1989). Heart Disease [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C52P4X.

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight

# Load dataset
heart_disease = fetch_ucirepo(id=45)
X = heart_disease.data.features
y = heart_disease.data.targets

print(y)

# Reduce to binary classification. Please see source for info, essentially there are 3 classes of "presence", but for now I will reduce to binary classification, absence vs presence
y = y.iloc[:, 0]
y = (y > 0).astype(int)

# Here, need to handle categorical data
# For example, cp has 4 options, 0,1,2,3 but need to expand this to individual categories
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print(X_encoded.head())

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, stratify=y, random_state=42
)

# Scale features - then use the scaling data in Vehicle
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))
print("Class weights:", class_weight_dict)

# Build model - note that output can't be softmax or sigmoid due to issues with Vehicle, so just have Dense(2) output
model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(16, activation='relu'),
    Dense(2)
])

# Compile model
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Use class_weight_dict to handle any class imbalance in the dataset
model.fit(X_train_scaled, y_train, epochs=50, batch_size=16, validation_split=0.1, class_weight=class_weight_dict)

logits = model.predict(X_test_scaled)
y_pred = np.argmax(logits, axis=1)
accuracy = np.mean(y_pred == y_test)
print("Test accuracy:", accuracy)




     num
0      0
1      2
2      1
3      0
4      0
..   ...
298    1
299    2
300    3
301    1
302    0

[303 rows x 1 columns]
   age  trestbps  chol  thalach  oldpeak  sex_1   cp_2   cp_3   cp_4  fbs_1  \
0   63       145   233      150      2.3   True  False  False  False   True   
1   67       160   286      108      1.5   True  False  False   True  False   
2   67       120   229      129      2.6   True  False  False   True  False   
3   37       130   250      187      3.5   True  False   True  False  False   
4   41       130   204      172      1.4  False   True  False  False  False   

   restecg_1  restecg_2  exang_1  slope_2  slope_3  ca_1.0  ca_2.0  ca_3.0  \
0      False       True    False    False     True   False   False   False   
1      False       True     True     True    False   False   False    True   
2      False       True     True     True    False   False    True   False   
3      False      False    False    False     True   False   False   False   
4  

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - accuracy: 0.5717 - loss: 0.6904 - val_accuracy: 0.8400 - val_loss: 0.5657
Epoch 2/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7005 - loss: 0.5872 - val_accuracy: 0.8800 - val_loss: 0.5260
Epoch 3/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7208 - loss: 0.5384 - val_accuracy: 0.8000 - val_loss: 0.4910
Epoch 4/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8038 - loss: 0.4804 - val_accuracy: 0.8000 - val_loss: 0.4563
Epoch 5/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8396 - loss: 0.4315 - val_accuracy: 0.7600 - val_loss: 0.4312
Epoch 6/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8179 - loss: 0.4338 - val_accuracy: 0.7600 - val_loss: 0.4058
Epoch 7/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8213 - loss: 0.4180 - val_accuracy: 0.8400 - val_loss: 0.3828
Epoch 8/50
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8671 - loss: 0.3804 - val_accuracy: 0.8400 - val_los

In [24]:
from sklearn.metrics import confusion_matrix
import numpy as np

# Predict probabilities
y_pred_prob = model.predict(X_test_scaled)

y_pred = np.argmax(y_pred_prob, axis=1)

cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
True Negatives: 28
False Positives: 5
False Negatives: 5
True Positives: 23


In [25]:
# Create the ONNX file
tf_model_func = tf.function(model)

# Input signature
spec = [tf.TensorSpec([None, X_train_scaled.shape[1]], tf.float32, name="input")]

# Convert to ONNX
onnx_model, _ = tf2onnx.convert.from_function(tf_model_func, input_signature=spec, opset=13)

with open("heart_disease_model.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("ONNX model exported successfully!")

ERROR:tf2onnx.tfonnx:rewriter <function rewrite_constant_fold at 0x7bc0c4679940>: exception `np.cast` was removed in the NumPy 2.0 release. Use `np.asarray(arr, dtype=dtype)` instead.


ONNX model exported successfully!


In [26]:
from google.colab import files

# Save the model first (in Colab's temporary space)
onnx_file = "heart_disease_model.onnx"
with open(onnx_file, "wb") as f:
    f.write(onnx_model.SerializeToString())

# Trigger download
files.download(onnx_file)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
unique, counts = np.unique(y_train, return_counts=True)
class_counts = dict(zip(unique, counts))
print(class_counts)


{np.int64(0): np.int64(131), np.int64(1): np.int64(111)}


In [28]:
np.set_printoptions(precision=2, suppress=True)
print("Means:", scaler.mean_)
print("Scales:", scaler.scale_)


Means: [ 54.55 130.96 249.84 149.96   1.     0.68   0.17   0.26   0.48   0.14
   0.     0.49   0.33   0.45   0.07   0.22   0.12   0.05   0.05   0.39]
Scales: [ 8.98 17.59 52.74 22.64  1.12  0.47  0.38  0.44  0.5   0.35  0.06  0.5
  0.47  0.5   0.25  0.41  0.32  0.22  0.21  0.49]


In [29]:
feature_mins = X_train.min(axis=0)
feature_maxs = X_train.max(axis=0)

print("Min values:\n", list(feature_mins.values))
print("\nMax values:\n", list(feature_maxs.values))


Min values:
 [np.int64(29), np.int64(94), np.int64(126), np.int64(71), np.float64(0.0), np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_, np.False_]

Max values:
 [np.int64(77), np.int64(200), np.int64(564), np.int64(202), np.float64(6.2), np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_, np.True_]
